# XGBoost Classification: Heart Disease

This notebook demonstrates a complete **binary classification** workflow with XGBoost. It uses the repository's UCI heart-disease dataset and predicts whether a patient has heart disease (`num > 0`) or not (`num = 0`). This is an educational example, not a medical diagnostic tool.

**Workflow:** understand the data -> define the target -> split into train/test sets -> preprocess missing and categorical values -> train XGBoost -> evaluate on unseen data -> inspect feature importance -> make a sample prediction.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
print(f"XGBoost version: {xgb.__version__}")

## 1. Load and understand the data

The original `num` field ranges from 0 to 4. For this binary task, 0 means no detected heart disease and any value above 0 means disease is present. We exclude `id` and `dataset` because they identify a row or data source rather than describing the patient.

In [ ]:
data_path_candidates = [
    Path("classification") / "heart_disease_uci.csv",
    Path("..") / "classification" / "heart_disease_uci.csv",
]
data_path = next((path for path in data_path_candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Could not find classification/heart_disease_uci.csv. Run this notebook "
        "from the repository root or the 'xg boost' folder."
    )

data = pd.read_csv(data_path)
if "num" not in data.columns:
    raise ValueError("Expected the target column 'num' in the heart-disease dataset.")

# Rows without a target cannot be used for supervised training.
data = data.dropna(subset=["num"]).copy()
data["target"] = (data["num"] > 0).astype(int)
excluded_columns = {"id", "dataset", "num", "target"}
feature_columns = [column for column in data.columns if column not in excluded_columns]
if not feature_columns:
    raise ValueError("No usable feature columns were found.")

X = data[feature_columns]
y = data["target"]
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print(f"Loaded {len(data)} rows from {data_path}")
print(f"Features: {len(feature_columns)} ({len(numeric_features)} numeric, {len(categorical_features)} categorical)")
print("Target counts (0 = no disease, 1 = disease):")
display(y.value_counts().sort_index().rename(index={0: "No disease", 1: "Disease"}).to_frame("count"))
print("Missing values per feature (largest counts first):")
display(X.isna().sum().sort_values(ascending=False).head(15).to_frame("missing values"))
display(data.head())

## 2. Create a stratified train/test split

The test set is held back until final evaluation. Stratification preserves the approximate disease/no-disease ratio in both sets. Preprocessing is fitted inside the training pipeline, which prevents information from the test set leaking into training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print("Training class proportions:")
display(y_train.value_counts(normalize=True).sort_index().rename(index={0: "No disease", 1: "Disease"}).to_frame("proportion"))

## 3. Preprocess and train XGBoost

XGBoost requires numeric inputs. The preprocessing pipeline fills numeric missing values with the training median, fills categorical missing values with the most frequent training value, and one-hot encodes categories. The pipeline applies the same transformations at prediction time.

In [ ]:
numeric_preprocessing = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_preprocessing = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessing, numeric_features),
        ("categorical", categorical_preprocessing, categorical_features),
    ],
)

model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
classifier = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model),
])

classifier.fit(X_train, y_train)
print("Training complete.")

## 4. Evaluate on the held-out test set

Accuracy is the share of correct predictions. Precision measures how many predicted disease cases are correct; recall measures how many actual disease cases are detected; F1 balances precision and recall. ROC-AUC measures ranking quality across classification thresholds. For health-related tasks, review recall and precision alongside accuracy, especially when classes are imbalanced.

In [ ]:
y_pred = classifier.predict(X_test)
y_probability = classifier.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1 score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_probability),
}
display(pd.DataFrame.from_dict(metrics, orient="index", columns=["test score"]).round(3))

print("Classification report (0 = no disease, 1 = disease):")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1],
    target_names=["No disease", "Disease"],
    zero_division=0,
))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["No disease", "Disease"],
    cmap="Blues",
)
plt.title("XGBoost: test-set confusion matrix")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_probability)
plt.title("XGBoost: test-set ROC curve")
plt.tight_layout()
plt.show()

## 5. Inspect feature importance and make a prediction

XGBoost's built-in importance is based on how the fitted trees use each transformed feature; it is useful for exploration but does not establish cause and effect. The final example predicts one held-out patient and reports the estimated probability for the disease class.

In [ ]:
preprocessor_fitted = classifier.named_steps["preprocessor"]
model_fitted = classifier.named_steps["model"]
transformed_feature_names = preprocessor_fitted.get_feature_names_out()
feature_importance = (
    pd.Series(model_fitted.feature_importances_, index=transformed_feature_names)
    .sort_values(ascending=False)
    .head(15)
    .sort_values()
)
feature_importance.plot(kind="barh", figsize=(9, 6), title="Top XGBoost feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

sample_patient = X_test.iloc[[0]]
predicted_class = int(classifier.predict(sample_patient)[0])
disease_probability = float(classifier.predict_proba(sample_patient)[0, 1])
print(f"Predicted class: {predicted_class} (0 = no disease, 1 = disease)")
print(f"Estimated disease probability: {disease_probability:.1%}")
print(f"Actual class: {int(y_test.iloc[0])}")

## Cycle recap

1. Load and inspect labeled data.
2. Convert the original target into a binary label and remove identifier/source columns.
3. Split into training and test sets with stratification.
4. Learn missing-value handling and categorical encoding from training data.
5. Fit an `XGBClassifier` to learn nonlinear decision rules and feature interactions.
6. Evaluate once on the held-out test set using multiple metrics and plots.
7. Inspect model feature importance and use the fitted pipeline for new rows.

For a real application, use an appropriate validation strategy, tune the decision threshold for the intended use, assess calibration and subgroup performance, and obtain clinical review. This notebook is for learning only.